# Launch in Google Colab

| | |
|---|---|
| **One click** | [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/02_run_meeting_llm.ipynb) |
| **Direct link** | https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/02_run_meeting_llm.ipynb |

**[▶ Open this notebook in Colab](https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/02_run_meeting_llm.ipynb)** (sign in with the Google account that has the hackathon Drive folder).

After it opens: **Runtime → Change runtime type → CPU** → add Colab secret **`GEMINI_API_KEY`** → **Runtime → Run all** (or run §1–§6 in order).

*Viewing on GitHub? Use the badge or link above — do not run the notebook on GitHub; it only runs in Colab.*

# Gemma 4 Good — Municipal Meeting Intelligence

**Real government PDFs + audio → Gemma 4** (OCR, token budget, policy reasoning, meeting drift).

**Hackathon story hook:** *What percentage of a small town’s revenue comes from speed traps?*  
Nationwide, Governing found **600+** jurisdictions where fines are **>10%** of the general fund and **284+** where they’re **>20%** ([Addicted to Fines](https://www.governing.com/archive/gov-addicted-to-fines.html)). This notebook grounds that question in **real county minutes + budgets** (default: Tuscaloosa County `county_01125`).

## Judges — quick start

1. Colab secret **`GEMINI_API_KEY`** ([AI Studio](https://aistudio.google.com/apikey))
2. **Runtime → Change runtime type → CPU** (default path uses **Google AI Studio** for Gatekeeper + demos — inference runs on Google’s servers, not the Colab GPU)
3. **§2** — set `SCOPE = "fast"` (or `medium` / `full`)
4. **Runtime → Run all** (or §1 → §6)
5. Outputs: `My Drive/CommunityOne/hackathons/2026_Gemma_4_Good/`

| After §6 | Look here |
|----------|----------|
| Gatekeeper | `00_logs/triage_report_*.json` |
| Visual OCR | `03_processed_outputs/02_gemma_json/…/*visual_ocr*` |
| Reasoning trace | `*.thinking.thoughts.md` |
| Audio drift | `policy_drift.json` + `policy_drift.mmd` |

<details><summary>Runtime, models & HF (optional)</summary>

- **CPU runtime (recommended):** With the default **Google AI Studio** backend, Colab only does Drive I/O, PDF text extraction, and API calls — a **GPU runtime is not used** and Colab may warn “connected to GPU but not utilizing it.” Use **CPU** unless you opt into local Hugging Face below.
- Default models (AI Studio): **Demo 1/2 PDF** `gemma-4-26b-a4b-it`; **Demo 3** `gemma-4-31b-it`; **Demo 4** auto-picks from `models.list()` (often **Gemini Flash** for audio when Gemma Edge is not on your key).
- **`.mp4` meetings:** Demo 4 segments video as `video/mp4` chunks (`GOVERNANCE_DEMO4_VIDEO_CHUNKS=1`, default). Set `0` for audio-only MP3 chunks.
- **GPU runtime only if:** `GOVERNANCE_GATEKEEPER_FORCE_HF=1` or `GOVERNANCE_LLM_BACKEND=huggingface` + `HF_TOKEN` for local E2B (~10GB on the Colab GPU).

</details>


## Appendix: model ids (optional)

Resolved ids print in §3.


## Run order

**§1** Bootstrap → **§2** set `SCOPE` → **§3** Install → **§4** Keys → **§5** Inventory → **§6** Pipeline.

**§7+** optional reruns (Demos 1, 3, 4 only — Demo 2 token budget runs inside **§6**). For a live slot, keep **`SCOPE = "fast"`** in §2.


## 1) Bootstrap (Drive + repo)

Mounts Drive, pulls latest `open-navigator`, resolves the pipeline folder on Drive. Prints **Repo** and **Governance data root** — confirm the hackathon path, not `/content/...`.


In [ ]:
# Find the open-navigator repo, refresh it from GitHub, then import the resolver.
# Order matters: any cached scripts.colab.* modules must be evicted BEFORE the
# import below, or git-reset --hard won't take effect this session.
import os
import shutil
import sys
from pathlib import Path

REPO_PATH = Path("/content/open-navigator")
EPHEMERAL_DATA_DIR = Path("/content/_ephemeral_colab_pipeline_shell")

# 1) Clone if missing.
if not REPO_PATH.is_dir():
    print(f"Cloning open-navigator into {REPO_PATH}...")
    rc = os.system(f"git clone https://github.com/getcommunityone/open-navigator.git {REPO_PATH}")
    if rc != 0:
        raise RuntimeError(f"git clone failed (exit {rc})")

os.environ.setdefault("OPEN_NAVIGATOR_ROOT", str(REPO_PATH))

# 2) git fetch + reset --hard origin/main BEFORE the first import, so Python loads
#    the latest colab_paths.py from disk (not a cached stale version).
RUN_GIT_UPDATE = True
if RUN_GIT_UPDATE and (REPO_PATH / ".git").is_dir():
    print("🔄 Fetching latest open-navigator from GitHub...")
    rc = os.system(
        f"cd {REPO_PATH} && git config core.hooksPath /dev/null "
        "&& git fetch origin && git reset --hard origin/main"
    )
    if rc != 0:
        print(f"⚠️  git update returned exit {rc} — continuing with whatever is on disk.")

# 3) Defensive cleanup of stale session state from prior runs.
#    Without this, re-running the cell silently uses cached old code / paths.
for _mod in [m for m in list(sys.modules) if m.startswith("scripts.colab") or m.startswith("scripts.utils.gdrive_paths")]:
    sys.modules.pop(_mod, None)
_stale_root = os.environ.pop("GOVERNANCE_PIPELINE_DATA_ROOT", None)
if _stale_root:
    print(f"   Cleared stale GOVERNANCE_PIPELINE_DATA_ROOT={_stale_root}")

# 4) Remove ephemeral /content shell if a previous notebook left a fake data root
#    notebook version (or PIPE.ensure_dirs) left it behind. Real data lives on
#    mounted Drive — this folder is on disposable runtime disk and confuses judges.
if EPHEMERAL_DATA_DIR.is_dir():
    print(f"   Removing stale ephemeral folder {EPHEMERAL_DATA_DIR} "
          "(real data lives on Drive — this is just an empty shell from a prior bug).")
    shutil.rmtree(EPHEMERAL_DATA_DIR, ignore_errors=True)

# 5) Put the repo on sys.path so `from scripts.colab.colab_paths import …` works.
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# 6) Now the first import sees the freshly-pulled code.
from scripts.colab.colab_paths import maybe_mount_google_drive, setup_notebook_paths

maybe_mount_google_drive()

try:
    PATHS = setup_notebook_paths()
except RuntimeError as e:
    # Resolver couldn't find the data root on Drive. Surface its diagnostic +
    # show what IS visible under the mount so the user can spot the typo.
    print("\n❌ setup_notebook_paths() failed:\n")
    print(str(e))
    print("\nCurrent /content/drive view:")
    for p in (Path("/content/drive"), Path("/content/drive/MyDrive"),
              Path("/content/drive/MyDrive/CommunityOne"),
              Path("/content/drive/MyDrive/CommunityOne/hackathons")):
        if p.is_dir():
            children = sorted([c.name for c in p.iterdir()])[:20]
            print(f"  {p}: {children}")
        else:
            print(f"  {p}: (missing)")
    raise

print("Runtime:", "Google Colab" if PATHS.in_colab else "Local")
print("Repo:                 ", PATHS.project_path)
print("Governance data root: ", PATHS.governance_pipeline_data)


## 2) Demo scope

Edit **`SCOPE`** in the next cell, then run it (and re-run **§4 → §6** if you change it later).

| `SCOPE` | Time | States | Jurisdictions | Meeting dates | PDFs / jur | Audio chunks | Contact images |
|--------|------|--------|---------------|---------------|------------|--------------|----------------|
| **`fast`** | ~45–75 min | 1 | 1 (`county_01125`) | **2** | **6** | **2 recordings** × 2 chunks | off |

**`fast`** Demo 2: up to **12 pages per PDF** (full minutes packets). Set in §2 via `GOVERNANCE_DEMO_MAX_PAGES_PER_PDF` when you run `apply_scope`.
| **`medium`** | ~35–50 min | 2 | 2 | 2 | 2 | 3 | 6 |
| **`full`** | ~90+ min | 4 | 4 | 3 | 3 | 4 | 12 |

**`fast`** defaults to **`AL/county/county_01125`** (Tuscaloosa County) when it has media; other scopes pick the richest jurisdiction per state.

Override in §2: `os.environ["GOVERNANCE_DEMO_JURISDICTION_SLUG"] = "county_01125"` (or another slug).


In [ ]:
# ── Edit these lines only ───────────────────────────────────────
SCOPE = "fast"  # fast → county_01125 (Tuscaloosa County) | medium | full
MEDIA_SCOPE = "video"  # all | pdf | audio | video  (test one modality at a time)
# ─────────────────────────────────────────────────────────────────

import sys

_colab = PATHS.project_path / "scripts" / "colab"
if str(_colab) not in sys.path:
    sys.path.insert(0, str(_colab))

from demo_scope import PRESETS, apply_scope
from pipeline_media_scope import SCOPES as MEDIA_SCOPES, apply_media_scope

if SCOPE not in PRESETS:
    raise ValueError(f"SCOPE must be one of {list(PRESETS)} — got {SCOPE!r}")
_media_key = MEDIA_SCOPE.strip().lower().replace("_only", "")
if _media_key not in MEDIA_SCOPES and MEDIA_SCOPE.strip().lower() not in (
    "pdf_only",
    "audio_only",
    "video_only",
):
    raise ValueError(
        f"MEDIA_SCOPE must be one of all|pdf|audio|video (or *_only) — got {MEDIA_SCOPE!r}"
    )

ACTIVE_SCOPE = apply_scope(SCOPE)
ACTIVE_MEDIA = apply_media_scope(MEDIA_SCOPE)
print(f"SCOPE = {SCOPE!r}  →  {ACTIVE_SCOPE.label}  ({ACTIVE_SCOPE.eta})")
print(f"MEDIA_SCOPE = {MEDIA_SCOPE!r}  →  {ACTIVE_MEDIA.label}")

In [ ]:
import os
import sys

# Repo path is fixed in cell 4 (git fetch + reset --hard ran there).
proj = PATHS.project_path

assert (proj / "prompts" / "policy_analysis_v1.md").is_file(), (
    f"Missing {proj}/prompts/policy_analysis_v1.md"
)

sys.path.insert(0, str(proj))
COLAB_DIR = proj / "scripts" / "colab"
sys.path.insert(0, str(COLAB_DIR))

# Drop stale Colab copies of scripts/colab/*.py so imports pick up a fresh git pull.
for _mod in list(sys.modules):
    if _mod in ("gatekeeper_triage", "governance_meeting_llm", "gemma_hf_backend"):
        del sys.modules[_mod]

# Pin the resolved Drive path so downstream modules read it from the env.
os.environ.setdefault("GOVERNANCE_PIPELINE_DATA_ROOT", str(PATHS.governance_pipeline_data))

from scripts.utils.gdrive_paths import GovernancePipelinePaths

PIPE = GovernancePipelinePaths.resolve()
# Create only the subfolders we will write into (03_processed_outputs/*). We do
# NOT create 01_raw_inputs here — if the resolver pointed somewhere wrong, an
# empty 01_raw_inputs would mask the bug.
for _sub in (PIPE.transcripts, PIPE.gemma_json, PIPE.human_summaries):
    _sub.mkdir(parents=True, exist_ok=True)

from governance_meeting_llm import (
    AUDIO_EXTS,
    PDF_EXTS,
    TOKEN_BUDGET_HIGH,
    TOKEN_BUDGET_LOW,
    call_google_genai_multimodal,
    transcribe_audio_with_gemma,
    TRANSCRIPTION_SUPPORTED_LANGUAGES,
    chunk_audio_ffmpeg,
    demo4_drift_output_complete,
    find_existing_audio_chunks,
    force_reprocess_outputs,
    policy_chunk_output_complete,
    read_json_file,
    text_output_complete,
    classify_pdf_page_heuristic,
    extract_pdf_digital_text,
    is_scanned_pdf,
    load_contacts_lookup,
    load_meeting_data_lookup,
    load_text_file,
    merge_contacts_by_jurisdiction,
    merge_meeting_data_by_jurisdiction,
    mime_for,
    mirror_output_path,
    parse_policy_analysis_response,
    policy_drift_summarize,
    render_pdf_pages,
    select_demo4_media,
    walk_raw_inputs,
    embed_text_with_gemma,
    cosine_similarity_matrix,
    shield_review_text,
)

PROMPT_PATH = proj / "prompts" / "policy_analysis_v1.md"
POLICY_PROMPT = load_text_file(PROMPT_PATH)
print("Loaded prompt chars:", len(POLICY_PROMPT))
print("Pipeline data root:", PIPE.root)


## 3) Install dependencies

Lightweight install only (**CPU runtime is fine** — default Gatekeeper + demos call **Google AI Studio**, not the Colab GPU).

**Gatekeeper defaults to Google AI Studio** (no ~10GB model download). `transformers` is installed in **§4** only when you enable Hugging Face (`GOVERNANCE_GATEKEEPER_FORCE_HF=1` or `GOVERNANCE_LLM_BACKEND=huggingface`).


In [ ]:
# poppler-utils for pdf2image (Gatekeeper). Skipped off Colab.
import shutil, subprocess, sys
if not shutil.which("pdftoppm"):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "poppler-utils"], check=False)
    except FileNotFoundError:
        pass

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "google-genai>=1.0", "librosa", "pymupdf", "pdf2image"
    ]
)
print(
    "Default: Google AI Studio (CPU runtime OK). "
    "If GOVERNANCE_GATEKEEPER_FORCE_HF=1, §4 installs transformers>=5.5.0 and loads E2B on GPU."
)

## 4) API keys & models

Uses caps from **§2** (`SCOPE`). Prints resolved Gemma ids for your API key.


In [ ]:
import os

if "SCOPE" not in globals():
    SCOPE = "fast"
    import sys as _sys
    _c = REPO / "scripts" / "colab"
    if str(_c) not in _sys.path:
        _sys.path.insert(0, str(_c))
    from demo_scope import apply_scope
    apply_scope(SCOPE)

import sys
if str(proj / "scripts" / "colab") not in sys.path:
    sys.path.insert(0, str(proj / "scripts" / "colab"))
# Judge-friendly defaults (edit here only if needed)
os.environ.setdefault("GOVERNANCE_MODE", "DEMO")
os.environ.setdefault("GOVERNANCE_LLM_BACKEND", "google")
os.environ.setdefault("GOVERNANCE_LOG_LLM_CATALOG", "0")
os.environ.setdefault("GOVERNANCE_ORGANIZE_MEETINGS", "1")
os.environ.setdefault("GOVERNANCE_DEMO_DATE_SCOPE", "1")
os.environ.setdefault("GOVERNANCE_DEMO_YEAR_SCOPE", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_DPI", "120")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_COPY_TO_TMP", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_LARGE_PDF_BYTES", "1500000")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_TEXT_FIRST", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_TEXT_MIN_CHARS", "80")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_FSYNC", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_API_TIMEOUT_SECONDS", "60")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_DIRECT", "0")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PDF_PAGES", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_SKIP_PDF_PROBE", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_MAX_MEETING_SESSIONS", "3")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_MAX_FILES", "12")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_RULES_ONLY", "1")
# Demo 4: Hugging Face E2B/E4B (native audio/video); Opus slices cached on Drive.
os.environ.setdefault("GOVERNANCE_DEMO4_USE_HF", "1")
os.environ.setdefault("GOVERNANCE_DEMO4_HF_MODEL", "google/gemma-4-E2B-it")
os.environ.setdefault("GOVERNANCE_DEMO4_MODEL", "")
os.environ.setdefault("GOVERNANCE_DEMO4_PREFER_OPUS", "1")
# Progress: wall-clock timestamps, step timing, heartbeat while HF/ffmpeg/generate run
os.environ.setdefault("GOVERNANCE_STEP_TIMESTAMPS", "1")
os.environ.setdefault("GOVERNANCE_STEP_TIMING", "1")
os.environ.setdefault("GOVERNANCE_HEARTBEAT_SECONDS", "45")
os.environ.setdefault("GOVERNANCE_PIPELINE_VERBOSE", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_PROGRESS", "1")
# Demo 4 video/mp4 chunks: set by MEDIA_SCOPE in §2 (video → 1, audio → 0, all → 1)
os.environ.setdefault("GOVERNANCE_DEMO4_VIDEO_CHUNKS", "1")
# os.environ.setdefault("GOVERNANCE_GATEKEEPER_FORCE_HF", "1")  # opt-in: ~10GB HF download

LLM_BACKEND = os.environ.get("GOVERNANCE_LLM_BACKEND", "google").strip().lower()
USE_HF_ALL = LLM_BACKEND in ("huggingface", "hf", "local")

try:
    from google.colab import userdata
    _get_secret = userdata.get
except ImportError:
    def _get_secret(name: str):
        return os.environ.get(name)


def _sanitize_key(value):
    if not value:
        return value
    cleaned = (
        value.strip().replace("\r", "").replace("\n", "").replace("\t", "").replace(" ", "")
    )
    if cleaned != value.strip():
        print(f"   ⚠️  API key whitespace stripped — re-copy Colab Secret to silence this.")
    return cleaned


HF_TOKEN = _sanitize_key(_get_secret("HF_TOKEN"))
GEMINI_API_KEY = _sanitize_key(_get_secret("GEMINI_API_KEY") or _get_secret("GOOGLE_API_KEY"))

if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY (Colab Secret). https://aistudio.google.com/apikey")
if not GEMINI_API_KEY.startswith("AIza") or len(GEMINI_API_KEY) < 35:
    raise RuntimeError(f"GEMINI_API_KEY looks malformed (len={len(GEMINI_API_KEY)}).")
API_KEY = GEMINI_API_KEY

GENAI_MODEL = os.environ.get("GOVERNANCE_GENAI_MODEL", "gemma-4-26b-a4b-it").strip()
THINKING_MODEL = os.environ.get("GOVERNANCE_THINKING_MODEL", "gemma-4-31b-it").strip()
GATEKEEPER_MODEL = os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", "gemma-4-e2b-it").strip()
EMBEDDING_MODEL = os.environ.get("GOVERNANCE_EMBEDDING_MODEL", "embeddinggemma-300m").strip()
SHIELD_MODEL = os.environ.get("GOVERNANCE_SHIELD_MODEL", "shieldgemma-9b").strip() or GENAI_MODEL

from gemma_hf_backend import (
    gatekeeper_use_huggingface,
    model_requires_huggingface,
    preload_gemma_hf,
    print_hf_model_catalog,
    resolve_hf_model_id,
    ensure_transformers_gemma4_ready,
    _HF_GATEKEEPER_REPO_DEFAULT,
)

NEEDS_HF = (
    USE_HF_ALL
    or model_requires_huggingface(GENAI_MODEL)
    or gatekeeper_use_huggingface()
)
if NEEDS_HF:
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN required when GOVERNANCE_GATEKEEPER_FORCE_HF=1 or GOVERNANCE_LLM_BACKEND=huggingface. "
            "Default Gatekeeper uses AI Studio only — no HF_TOKEN needed."
        )
    os.environ["HF_TOKEN"] = HF_TOKEN

_log_llm_catalog = os.environ.get("GOVERNANCE_LOG_LLM_CATALOG", "0") == "1"

from gatekeeper_triage import (
    _GEMMA_HEAVY_FALLBACKS,
    _GEMMA_THINKING_FALLBACKS,
    _GEMMA_DEMO4_AUDIO_FALLBACKS,
    _GEMMA_GATEKEEPER_AI_FALLBACKS,
    _GEMMA_TRIAGE_FALLBACKS,
    _build_genai_client,
    print_available_models,
    resolve_model_id,
)

_resolver_client = _build_genai_client(GEMINI_API_KEY)
if _log_llm_catalog:
    print_available_models(
        _resolver_client,
        requested=(GENAI_MODEL, GATEKEEPER_MODEL, EMBEDDING_MODEL, SHIELD_MODEL),
        role="Google AI Studio",
    )

if model_requires_huggingface(GENAI_MODEL) or model_requires_huggingface(THINKING_MODEL):
    GENAI_MODEL = resolve_hf_model_id(GENAI_MODEL)
    THINKING_MODEL = resolve_hf_model_id(THINKING_MODEL)
    ensure_transformers_gemma4_ready(auto_install=True)
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GENAI_MODEL, THINKING_MODEL), role="Hugging Face (demos)")
    print(f"Loading HF weights for demos: {GENAI_MODEL} …")
    preload_gemma_hf(GENAI_MODEL, load_audio_variant=True)
    if THINKING_MODEL != GENAI_MODEL:
        print(f"Loading HF weights for Demo 3 thinking: {THINKING_MODEL} …")
        preload_gemma_hf(THINKING_MODEL, load_audio_variant=True)
else:
    GENAI_MODEL = resolve_model_id(
        _resolver_client, GENAI_MODEL, fallbacks=_GEMMA_HEAVY_FALLBACKS, role="Demos 1/2 PDF (AI Studio)"
    )
    THINKING_MODEL = resolve_model_id(
        _resolver_client,
        THINKING_MODEL,
        fallbacks=_GEMMA_THINKING_FALLBACKS,
        role="Demo 3 thinking (AI Studio)",
    )
if gatekeeper_use_huggingface():
    ensure_transformers_gemma4_ready(auto_install=True)
    hf_gate = (
        os.environ.get("GOVERNANCE_GATEKEEPER_MODEL", "").strip()
        or os.environ.get("GOVERNANCE_HF_GATEKEEPER_MODEL", "").strip()
        or _HF_GATEKEEPER_REPO_DEFAULT
    )
    GATEKEEPER_MODEL = resolve_hf_model_id(hf_gate)
    if _log_llm_catalog:
        print_hf_model_catalog(requested=(GATEKEEPER_MODEL,), role="Hugging Face (Gatekeeper)")
    print(f"Loading HF weights for Gatekeeper: {GATEKEEPER_MODEL} … (~10GB first time)")
    preload_gemma_hf(GATEKEEPER_MODEL, load_audio_variant=False)
else:
    _gate_requested = GATEKEEPER_MODEL
    GATEKEEPER_MODEL = resolve_model_id(
        _resolver_client,
        GATEKEEPER_MODEL,
        fallbacks=_GEMMA_GATEKEEPER_AI_FALLBACKS,
        role="Gatekeeper (AI Studio)",
    )
    if GATEKEEPER_MODEL != _gate_requested:
        print(
            f"  Gatekeeper model: {_gate_requested!r} not on this key — using {GATEKEEPER_MODEL!r}"
        )
    if any(x in GATEKEEPER_MODEL for x in ("26b", "31b", "27b")):
        print("  (heavier Gatekeeper model — yes/no triage may take 1–3 min per file)")

EMBEDDING_MODEL = resolve_model_id(
    _resolver_client,
    EMBEDDING_MODEL,
    fallbacks=("embeddinggemma-300m", "text-embedding-004", "gemini-embedding-001"),
    role="Embedding model",
)
SHIELD_MODEL = resolve_model_id(
    _resolver_client,
    SHIELD_MODEL,
    fallbacks=("shieldgemma-9b", "shieldgemma-2b"),
    role="Shield model (safety review)",
)

from colab_timed_steps import configure_pipeline_logging, heartbeat_interval
configure_pipeline_logging()
print(f"Pipeline logging: heartbeats every {heartbeat_interval():.0f}s (GOVERNANCE_HEARTBEAT_SECONDS)")

from gemma_hf_backend import demo4_use_huggingface, ensure_demo4_hf_ready, resolve_hf_model_id

if demo4_use_huggingface() and ACTIVE_MEDIA.run_demo4 and not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN required for Demo 4 (Gemma E2B/E4B audio/video on Hugging Face). "
        "Colab Secret HF_TOKEN + accept license at https://huggingface.co/google/gemma-4-E2B-it"
    )

if demo4_use_huggingface():
    DEMO4_MODEL = ensure_demo4_hf_ready()
    print(f"  Demo 4: Hugging Face {DEMO4_MODEL!r} (native audio/video — weights cached on Drive)")
elif model_requires_huggingface(GENAI_MODEL) or model_requires_huggingface(THINKING_MODEL):
    _d4_hf = os.environ.get("GOVERNANCE_DEMO4_MODEL", "").strip() or THINKING_MODEL
    DEMO4_MODEL = resolve_hf_model_id(_d4_hf)
else:
    from governance_meeting_llm import resolve_demo4_genai_model, list_demo4_capable_models_on_key
    _d4_requested = os.environ.get("GOVERNANCE_DEMO4_MODEL", "").strip()
    _d4_capable = list_demo4_capable_models_on_key(API_KEY, thinking_model=THINKING_MODEL)
    if _d4_capable:
        print("  Demo 4 audio-capable on this key:", ", ".join(_d4_capable[:8]))
    else:
        print("  Demo 4: no audio model on API key — set GOVERNANCE_DEMO4_USE_HF=1 (default)")
    DEMO4_MODEL = resolve_demo4_genai_model(
        GENAI_MODEL,
        gatekeeper_model=GATEKEEPER_MODEL,
        thinking_model=THINKING_MODEL,
        client=_resolver_client,
        api_key=API_KEY,
    )
    if _d4_requested and DEMO4_MODEL != _d4_requested:
        print(
            f"  Demo 4 model: {_d4_requested!r} not on this key — using {DEMO4_MODEL!r}"
        )

MAX_PDFS_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_PDFS_PER_JUR", "3"))
MAX_PAGES_PER_PDF = int(os.environ.get("GOVERNANCE_DEMO_MAX_PAGES_PER_PDF", "8"))
MAX_AUDIO_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_PER_JUR", "1"))
MAX_AUDIO_CHUNKS = int(os.environ.get("GOVERNANCE_DEMO_MAX_AUDIO_CHUNKS", "4"))
THINKING_BUDGET = int(os.environ.get("GOVERNANCE_DEMO_THINKING_BUDGET", "-1"))
DRIFT_FOCUS = os.environ.get("GOVERNANCE_DRIFT_FOCUS", "").strip() or None
_gate_max_env = os.environ.get("GOVERNANCE_GATEKEEPER_MAX_FILES", "").strip()
GATEKEEPER_MAX_FILES = int(_gate_max_env) if _gate_max_env else None


def _backend_label(model: str, *, gatekeeper: bool = False, demo4: bool = False) -> str:
    from gemma_hf_backend import gatekeeper_use_huggingface, demo4_use_huggingface, model_requires_huggingface
    if gatekeeper:
        return "Hugging Face" if gatekeeper_use_huggingface() else "Google AI Studio"
    if demo4 and demo4_use_huggingface():
        return "Hugging Face (E2B audio/video)"
    return "Hugging Face" if model_requires_huggingface(model) else "Google AI Studio"


print("LLM mode:", "all Hugging Face" if USE_HF_ALL else "hybrid (AI Studio default)")
if demo4_use_huggingface() and ACTIVE_MEDIA.run_demo4:
    print("Colab runtime: GPU required for Demo 4 (Hugging Face Gemma E2B ~10GB VRAM).")
elif not gatekeeper_use_huggingface() and not model_requires_huggingface(GENAI_MODEL):
    print(
        "Colab runtime: CPU OK for Gatekeeper + PDF demos (Google AI). "
        "Use GPU when Demo 4 runs (GOVERNANCE_DEMO4_USE_HF=1)."
    )
print("GenAI model (demos 1/2 PDF):", GENAI_MODEL, f"({_backend_label(GENAI_MODEL)})")
print("Thinking model (demo 3):  ", THINKING_MODEL, f"({_backend_label(THINKING_MODEL)})")
print("Audio model (demo 4):     ", DEMO4_MODEL, f"({_backend_label(DEMO4_MODEL, demo4=True)})")
print(
    "Gatekeeper (triage):     ",
    GATEKEEPER_MODEL,
    f"({_backend_label(GATEKEEPER_MODEL, gatekeeper=True)})",
)
if gatekeeper_use_huggingface():
    print("  (GOVERNANCE_GATEKEEPER_FORCE_HF=1 — local E2B weights)")
print("Embedding (demo 6):      ", EMBEDDING_MODEL, "(Google AI Studio)")
print("Shield (safety pass):    ", SHIELD_MODEL, "(Google AI Studio)")
print(
    "Safety review:           ",
    "on (end of §6)" if os.environ.get("GOVERNANCE_SAFETY_REVIEW", "1").strip().lower()
    not in ("0", "false", "no", "off") else "off",
)
# Optional §3b — probe Google AI Studio (text stream hack + audio modality)
# os.environ["GOVERNANCE_DEMO4_USE_HF"] = "0"
# os.environ["GOVERNANCE_DEMO4_TRY_A4B_AUDIO"] = "1"
# os.environ["GOVERNANCE_DEMO4_MODEL"] = "gemma-4-26b-a4b-it"
# from governance_meeting_llm import print_probe_google_gemma_studio
# _probe_audio = next(iter(INVENTORY.audio), None) if "INVENTORY" in dir() else None
# print_probe_google_gemma_studio(GEMINI_API_KEY, audio_path=_probe_audio)

print(
    "Caps:",
    f"pdfs/jur={MAX_PDFS_PER_JUR}",
    f"pages/pdf={MAX_PAGES_PER_PDF}",
    f"audio/jur={MAX_AUDIO_PER_JUR}",
    f"chunks/audio={MAX_AUDIO_CHUNKS}",
    f"thinking_budget={THINKING_BUDGET}",
    f"gatekeeper_max_files={GATEKEEPER_MAX_FILES if GATEKEEPER_MAX_FILES is not None else 'unlimited'}",
)


## 5) Inventory

Edit `SCOPE` in §2. §6 runs Gatekeeper and demos 1–4 for each selected place.

On Colab, scoped jurisdictions are **mirrored to `/content/…`** when missing or stale (Drive `_manifest.json` mtime). §6 then reads local disk (much faster). Set `GOVERNANCE_LOCAL_RAW_MIRROR=0` to stay on Drive only.


In [ ]:
from pathlib import Path
import os

DRIVE_RAW_ROOT = Path(
    os.environ.get("GOVERNANCE_RAW_INPUTS_ROOT", "").strip() or (PIPE.root / "01_raw_inputs")
).expanduser()
RAW_ROOT = DRIVE_RAW_ROOT
os.environ.setdefault("GOVERNANCE_LOCAL_RAW_MIRROR", "1")
os.environ.setdefault("GOVERNANCE_STEP_TIMING", "1")
os.environ.setdefault("GOVERNANCE_STEP_TIMESTAMPS", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_VERBOSE", "1")
os.environ.setdefault("GOVERNANCE_PIPELINE_PROGRESS", "1")
os.environ.setdefault("GOVERNANCE_GATEKEEPER_PROGRESS", "1")
PROCESSED_ROOT = PIPE.root / "03_processed_outputs"
GEMMA_JSON_ROOT = PROCESSED_ROOT / "02_gemma_json"
SUMMARIES_ROOT = PROCESSED_ROOT / "03_human_summaries"
SCRATCH_AUDIO_ROOT = PROCESSED_ROOT / "_scratch_audio_chunks"

try:
    from gemma_hf_backend import configure_hf_hub_cache, demo4_use_huggingface
    _hf_cache = os.environ.get("GOVERNANCE_HF_CACHE_DIR", "").strip()
    HF_CACHE_ROOT = Path(_hf_cache) if _hf_cache else (PROCESSED_ROOT / "_hf_hub_cache")
    configure_hf_hub_cache(HF_CACHE_ROOT)
    if demo4_use_huggingface():
        print(f"HF hub cache: {HF_CACHE_ROOT} (persists weights across Colab sessions)")
except ImportError:
    pass

for p in (GEMMA_JSON_ROOT, SUMMARIES_ROOT, SCRATCH_AUDIO_ROOT):
    p.mkdir(parents=True, exist_ok=True)

if not DRIVE_RAW_ROOT.is_dir():
    raise FileNotFoundError(
        f"Raw inputs root missing: {DRIVE_RAW_ROOT}. Run 01_copy_scraped_meetings_cache_to_gdrive.py "
        "or set GOVERNANCE_RAW_INPUTS_ROOT."
    )

from demo_scope import get_active_preset, filter_inventories_for_scope, print_scope_plan

try:
    from meeting_date_scope import resolve_demo_meeting_dates_limit
    _demo_date_cap = resolve_demo_meeting_dates_limit()
except ImportError:
    _demo_date_cap = None

_DEMO_SCOPE = get_active_preset()
from colab_timed_steps import timed_step

with timed_step("Inventory walk on Drive (discover jurisdictions)"):
    _ALL_INVENTORIES = [inv for inv in walk_raw_inputs(DRIVE_RAW_ROOT) if inv.has_media]
    INVENTORIES = filter_inventories_for_scope(_ALL_INVENTORIES, _DEMO_SCOPE)

print(f"Drive raw inputs: {DRIVE_RAW_ROOT}")
print_scope_plan(_DEMO_SCOPE, _ALL_INVENTORIES, INVENTORIES)

if INVENTORIES:
    from colab_local_raw_mirror import mirror_inventories_to_local_raw

    INVENTORIES, RAW_ROOT = mirror_inventories_to_local_raw(
        INVENTORIES, DRIVE_RAW_ROOT
    )
print(f"Pipeline raw inputs (§6): {RAW_ROOT}")

if not INVENTORIES:
    print(
        "\nNo media found. Drop PDFs / audio under "
        f"{RAW_ROOT}/<STATE>/<county|municipality>/<jurisdiction_slug>/…"
    )


## 6) Run pipeline

End-to-end per jurisdiction. **`SCOPE`** in §2 sets how many states/places run.

Live log: `00_logs/gatekeeper_<jurisdiction>_*.log` — after the jurisdiction banner, §6 prints **walk / classify / API** steps (5–10 min on Drive is normal before the first Gemma call).


In [ ]:
import importlib
import os

import colab_demos
import jurisdiction_pipeline as _jp
importlib.reload(colab_demos)
importlib.reload(_jp)

from colab_demos import DemoContext
from governance_meeting_llm import inventory_for_jurisdiction
from jurisdiction_pipeline import JurisdictionRunContext, run_governance_pipeline

if not INVENTORIES:
    print("No inventories — skip pipeline.")
else:
    _demo_ctx = DemoContext(
        api_key=API_KEY,
        genai_model=GENAI_MODEL,
        thinking_model=THINKING_MODEL,
        demo4_model=DEMO4_MODEL,
        gatekeeper_model=GATEKEEPER_MODEL,
        raw_root=RAW_ROOT,
        processed_root=PROCESSED_ROOT,
        gemma_json_root=GEMMA_JSON_ROOT,
        summaries_root=SUMMARIES_ROOT,
        scratch_audio_root=SCRATCH_AUDIO_ROOT,
        policy_prompt=POLICY_PROMPT,
        max_pdfs_per_jur=MAX_PDFS_PER_JUR,
        max_pages_per_pdf=MAX_PAGES_PER_PDF,
        max_audio_per_jur=MAX_AUDIO_PER_JUR,
        max_audio_chunks=MAX_AUDIO_CHUNKS,
        thinking_budget=THINKING_BUDGET,
        drift_focus=DRIFT_FOCUS,
    )
    _run_ctx = JurisdictionRunContext(
        raw_root=RAW_ROOT,
        pipe_root=PIPE.root,
        api_key=API_KEY,
        gatekeeper_model=GATEKEEPER_MODEL,
        demo_ctx=_demo_ctx,
        shield_model=SHIELD_MODEL,
        demo_date_cap=_demo_date_cap,
        gatekeeper_max_files=GATEKEEPER_MAX_FILES,
        run_safety_review=os.environ.get("GOVERNANCE_SAFETY_REVIEW", "1").strip().lower()
        not in ("0", "false", "no", "off"),
    )
    run_governance_pipeline(INVENTORIES, _run_ctx)
    INVENTORIES = [
        inventory_for_jurisdiction(RAW_ROOT, inv.jurisdiction.root) or inv
        for inv in INVENTORIES
    ]
    if _demo_date_cap:
        from meeting_date_scope import filter_inventory_media
        for inv in INVENTORIES:
            inv.pdfs, inv.audio = filter_inventory_media(
                inv.pdfs, inv.audio, RAW_ROOT, inv.jurisdiction.root, max_dates=_demo_date_cap
            )
    from pipeline_media_scope import apply_media_scope_to_inventory
    for inv in INVENTORIES:
        apply_media_scope_to_inventory(inv)
    print("\nPipeline complete (demos 1–4). Optional cells below use refreshed inventories.")


## 7) Demo 1 — Scanned PDFs *(optional rerun)*

Already run in **§6** (pipeline). Re-run only to regenerate OCR for capped PDFs.


In [ ]:
# Demo 1 — Native multimodality / visual document parsing.
# Probes the digital-text layer of every PDF and uses Gemma 4 to OCR scans.
from datetime import datetime, timezone

DEMO1_SYSTEM = (
    "You are a careful document transcription engine. Faithfully transcribe every "
    "word and number on each page of the attached PDF in reading order. Preserve "
    "table structure with vertical bars and dashes. Do not paraphrase, summarize, "
    "or invent content."
)
DEMO1_USER = (
    "Transcribe the attached PDF page by page. Begin each page with a heading line "
    "'### Page <n>' on its own line. If a page is blank, write '(blank page)'."
)

demo1_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdfs = inv.pdfs[:MAX_PDFS_PER_JUR]
    if not pdfs:
        continue
    print(f"\n— {j.relative_label} — {len(pdfs)} PDF(s)")
    for pdf in pdfs:
        try:
            digital_chars = len(extract_pdf_digital_text(pdf))
        except Exception as e:
            print(f"  ! {pdf.name}: PDF probe failed — {e}")
            continue
        scanned = digital_chars < 200
        tag = "SCANNED (dark data)" if scanned else f"digital ({digital_chars} chars)"
        print(f"  • {pdf.name}: {tag}")

        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO1_SYSTEM,
                user_text=DEMO1_USER,
                media=[(pdf, "application/pdf")],
                temperature=0.0,
                max_output_tokens=8192,
                media_resolution=TOKEN_BUDGET_HIGH if scanned else None,
            )
        except Exception as e:
            print(f"    ! Gemma call failed: {e}")
            continue

        out_txt = mirror_output_path(
            input_path=pdf,
            raw_root=RAW_ROOT,
            processed_root=GEMMA_JSON_ROOT,
            suffix=".visual_ocr.txt",
        )
        out_txt.write_text(result.text or "(empty response)", encoding="utf-8")
        demo1_report.append(
            {
                "jurisdiction": j.relative_label,
                "fips": j.fips,
                "pdf": str(pdf.relative_to(RAW_ROOT)),
                "scanned": scanned,
                "digital_chars": digital_chars,
                "output": str(out_txt.relative_to(PROCESSED_ROOT)),
                "model_chars": len(result.text or ""),
            }
        )
        print(f"    → {out_txt.relative_to(PROCESSED_ROOT)} ({len(result.text or '')} chars)")

scanned_count = sum(1 for r in demo1_report if r["scanned"])
print(
    f"\nDemo 1 done: {len(demo1_report)} PDFs, {scanned_count} flagged scanned "
    "→ Gemma 4 visual OCR recovered text that pdftotext could not."
)

import json as _json
demo1_summary_path = GEMMA_JSON_ROOT / "_demo1_visual_ocr_report.json"
demo1_summary_path.write_text(_json.dumps(demo1_report, indent=2), encoding="utf-8")
print("Report:", demo1_summary_path.relative_to(PROCESSED_ROOT))

## 8) Demo 3 — Policy deconstruction *(optional rerun)*

**Show judges:** `*.thinking.thoughts.md` next to the JSON — Gemma's planning on a real minutes PDF (e.g. Tuscaloosa nuisance demolitions: safety narrative vs. property-rights dissent).

Demo 3: **`THINKING_MODEL`** (`gemma-4-31b-it`). Demos 1/2 PDF: **`GENAI_MODEL`**. Demo 4: **`DEMO4_MODEL`** on **Hugging Face E2B/E4B** (`GOVERNANCE_DEMO4_USE_HF=1`, Opus chunks cached under `_scratch_audio_chunks`).


In [ ]:
# Demo 3 — Built-in thinking mode on the deconstruction prompt.
# One representative PDF per jurisdiction is processed with include_thoughts=True.
import json as _json

DEMO3_SYSTEM = (
    "You are an expert political scientist and data architect specializing in "
    "local governance. Follow the user's instructions exactly; preserve JSON validity."
)

_PRIORITY_PATTERNS = (
    "demolition", "demolitions", "nuisance",
    "minutes", "regular_session", "regular-session", "regular session",
    "council", "commission", "board", "hearing",
)


def _pick_representative_pdf(pdfs):
    if not pdfs:
        return None
    scored = []
    for p in pdfs:
        name = p.name.lower()
        score = 0
        for tag in _PRIORITY_PATTERNS:
            if tag in name:
                score += 5
        try:
            score += min(p.stat().st_size // 50_000, 50)
        except OSError:
            pass
        scored.append((score, p))
    scored.sort(key=lambda t: (-t[0], t[1].name))
    return scored[0][1]


demo3_report = []
for inv in INVENTORIES:
    j = inv.jurisdiction
    pdf = _pick_representative_pdf(inv.pdfs)
    if pdf is None:
        continue
    print(f"\n— {j.relative_label}: {pdf.name}")

    geo_hint = (
        f"Geography hint from folder layout: state_code={j.state_code}, "
        f"scope={j.scope}, fips_or_place_id={j.fips or 'unknown'}. "
        "Use this when populating county_fips / county / postal_code in each decision."
    )
    user_text = (
        f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n"
        "The attached PDF contains the meeting record. Apply the full deconstruction "
        "prompt to it. Stick to what is actually in the document."
    )

    try:
        result = call_google_genai_multimodal(
            api_key=API_KEY,
            model=GENAI_MODEL,
            system_instruction=DEMO3_SYSTEM,
            user_text=user_text,
            media=[(pdf, "application/pdf")],
            temperature=0.1,
            max_output_tokens=8192,
            media_resolution=TOKEN_BUDGET_HIGH,
            include_thoughts=True,
            thinking_budget=THINKING_BUDGET,
        )
    except Exception as e:
        print(f"  ! Gemma call failed: {e}")
        continue

    parsed = parse_policy_analysis_response(result.text or "")

    raw_out = mirror_output_path(
        input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
        suffix=".thinking.raw.txt",
    )
    raw_out.write_text(result.text or "", encoding="utf-8")

    if parsed.get("json_analysis") is not None:
        json_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.json",
        )
        json_out.write_text(
            _json.dumps(parsed["json_analysis"], indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"  → {json_out.relative_to(PROCESSED_ROOT)}")

    if parsed.get("summary"):
        summary_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=SUMMARIES_ROOT,
            suffix=".thinking.summary.md",
        )
        summary_out.write_text(parsed["summary"], encoding="utf-8")
        print(f"  → {summary_out.relative_to(PROCESSED_ROOT)}")

    if result.thoughts:
        thoughts_out = mirror_output_path(
            input_path=pdf, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".thinking.thoughts.md",
        )
        thoughts_out.write_text(result.thoughts, encoding="utf-8")
        print(
            f"  → {thoughts_out.relative_to(PROCESSED_ROOT)} "
            f"(reasoning trace: {len(result.thoughts)} chars)"
        )
    else:
        print("  (no thoughts returned — model may not have surfaced a trace this run)")

    demo3_report.append(
        {
            "jurisdiction": j.relative_label,
            "pdf": str(pdf.relative_to(RAW_ROOT)),
            "thoughts_chars": len(result.thoughts or ""),
            "json_ok": parsed.get("json_analysis") is not None and "_error" not in (parsed.get("json_analysis") or {}),
            "parse_error": parsed.get("parse_error"),
        }
    )

if demo3_report:
    ok = sum(1 for r in demo3_report if r["json_ok"])
    print(
        f"\nDemo 3 done: {len(demo3_report)} PDFs deconstructed, "
        f"{ok} produced parseable JSON. Reasoning trace saved alongside each output."
    )
else:
    print("\nDemo 3: no PDFs available — check the walker output above.")

## 9) Demo 4 — Audio drift *(optional rerun)*

Prefer **§6** (`run_governance_pipeline`) — it uses Hugging Face E2B + Opus chunks via `colab_demos.run_demo4`. This cell is a thin rerun only when §6 skipped Demo 4.

**Show judges:** `policy_drift.json` and `policy_drift.mmd` under each audio folder — how one issue moves across 15‑minute chunks of a long meeting.

Uses agenda+minutes text (when organized in §4) to ground names in the audio prompt. Requires `GOVERNANCE_DEMO4_USE_HF=1` (default), GPU, and `HF_TOKEN`. Optional focus: `GOVERNANCE_DRIFT_FOCUS`.


In [ ]:
# Demo 4 — Long-meeting chunking + Policy Drift Detector.
# Splits audio into 15-minute chunks, runs policy_analysis_v1 per chunk, then runs the drift pass.
import json as _json

DEMO4_SYSTEM = (
    "You are an expert political scientist analyzing one chunk of a long meeting. "
    "Follow the user's instructions exactly; preserve JSON validity. The chunk_index "
    "tells you which 15-minute slice of the meeting this audio covers."
)

try:
    from meeting_grouping import (
        build_meeting_collateral_brief,
        format_audio_analysis_prompt,
        resolve_meeting_dir,
    )
except ImportError:
    build_meeting_collateral_brief = None

_brief_cache = {}

demo4_report = []
_demo4_force = force_reprocess_outputs()
for inv in INVENTORIES:
    j = inv.jurisdiction
    audios = select_demo4_media(inv.audio, RAW_ROOT, max_files=MAX_AUDIO_PER_JUR)
    if not audios:
        continue
    print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
    for audio in audios:
        print(f"  • {audio.name}")
        per_audio_dir = mirror_output_path(
            input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
        )
        per_audio_dir.mkdir(parents=True, exist_ok=True)

        drift_out = per_audio_dir / "policy_drift.json"
        from governance_meeting_llm import (
            demo4_media_work_dir,
            demo4_prefer_opus_chunks,
            demo4_use_video_chunks,
            discover_demo4_chunk_media,
            find_existing_demo4_chunks,
            iter_demo4_ingest_strategies,
            model_supports_video_input,
        )

        scratch = demo4_media_work_dir(SCRATCH_AUDIO_ROOT, RAW_ROOT, audio)
        use_video = (
            audio.suffix.lower() in {".mp4", ".mov", ".webm", ".mkv"}
            and demo4_use_video_chunks()
            and model_supports_video_input(DEMO4_MODEL)
            and not demo4_prefer_opus_chunks()
        )
        existing_chunks = find_existing_demo4_chunks(scratch, video=use_video)
        if existing_chunks and not _demo4_force:
            chunk_media = discover_demo4_chunk_media(scratch, video=use_video)[:MAX_AUDIO_CHUNKS]
            print(f"    reuse {len(existing_chunks)} ffmpeg chunk(s) from scratch")
        else:
            ingest_plans = list(
                iter_demo4_ingest_strategies(
                    audio,
                    out_dir=scratch,
                    chunk_minutes=15,
                    prefer_video=use_video,
                    demo4_model=DEMO4_MODEL,
                )
            )
            if not ingest_plans:
                print("    ! ffmpeg could not produce audio slices (Opus or MP3)")
                continue
            _lbl, chunk_media = ingest_plans[0]
            chunk_media = chunk_media[:MAX_AUDIO_CHUNKS]
            print(f"    ingest {_lbl}: {len(chunk_media)} chunk(s) (cap MAX_AUDIO_CHUNKS={MAX_AUDIO_CHUNKS})")
        if not chunk_media:
            continue
        print(f"    {len(chunk_media)} chunk(s) of 15 min each (cap MAX_AUDIO_CHUNKS={MAX_AUDIO_CHUNKS})")

        chunk_jsons = []
        _chunks_regenerated = False
        for idx, (chunk_path, chunk_mime) in enumerate(chunk_media):
            chunk_out = per_audio_dir / f"chunk_{idx:03d}.json"
            if not _demo4_force and policy_chunk_output_complete(chunk_out):
                data = read_json_file(chunk_out) or {}
                chunk_jsons.append(data.get("json_analysis") or {})
                print(f"    chunk {idx}: reuse existing JSON")
                continue
            _chunks_regenerated = True
            geo_hint = (
                f"Geography hint: state_code={j.state_code}, scope={j.scope}, "
                f"fips_or_place_id={j.fips or 'unknown'}. chunk_index={idx} of {len(chunk_media)}."
            )
            meeting_dir = (
                resolve_meeting_dir(audio, RAW_ROOT) if build_meeting_collateral_brief else None
            )
            brief = ""
            if meeting_dir and build_meeting_collateral_brief:
                mk = str(meeting_dir)
                if mk not in _brief_cache:
                    _brief_cache[mk] = build_meeting_collateral_brief(
                        meeting_dir,
                        api_key=API_KEY,
                        model=GENAI_MODEL,
                        client=None,
                    )
                brief = _brief_cache.get(mk) or ""
            chunk_hint = (
                "The attached audio is one 15-minute slice of a longer governance meeting. "
                "Apply the deconstruction prompt to what is audible. Use the chunk_index "
                "to anchor the timeline and assign consistent subject_id slugs across chunks."
            )
            user_text = format_audio_analysis_prompt(
                policy_prompt=POLICY_PROMPT,
                meeting_brief=brief,
                geo_hint=geo_hint,
                chunk_hint=chunk_hint,
            ) if brief and build_meeting_collateral_brief else (
                f"{POLICY_PROMPT}\n\n---\n{geo_hint}\n\n{chunk_hint}"
            )
            try:
                result = call_google_genai_multimodal(
                    api_key=API_KEY,
                    model=DEMO4_MODEL,
                    system_instruction=DEMO4_SYSTEM,
                    user_text=user_text,
                    media=[(chunk_path, chunk_mime)],
                    temperature=0.1,
                    max_output_tokens=8192,
                    demo4=True,
                )
            except Exception as e:
                print(f"    ! chunk {idx} failed: {e}")
                continue
            parsed = parse_policy_analysis_response(result.text or "")
            chunk_out.write_text(
                _json.dumps(
                    {
                        "chunk_index": idx,
                        "audio_source": str(chunk_path.name),
                        "json_analysis": parsed.get("json_analysis"),
                        "summary": parsed.get("summary"),
                        "parse_error": parsed.get("parse_error"),
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )
            chunk_jsons.append(parsed.get("json_analysis") or {})
            print(f"    chunk {idx}: → {chunk_out.relative_to(PROCESSED_ROOT)}")

        if not chunk_jsons or not any(chunk_jsons):
            continue

        # Drift detector: Gemma 4 takes every chunk's JSON and reports per-subject
        # narrative drift along the five `narrative_analysis` dimensions defined in
        # `prompts/policy_analysis_v1.md` (dominant_narrative, dissenting_interpretations,
        # causal_interpretations, value_conflicts, tradeoff_analysis). The canonical
        # prompt body is pinned into context so the model honors the latest schema verbatim.
        if (
            not _demo4_force
            and not _chunks_regenerated
            and demo4_drift_output_complete(drift_out)
        ):
            drift = read_json_file(drift_out) or {}
            print(f"    drift detector: reuse existing → {drift_out.relative_to(PROCESSED_ROOT)}")
        else:
            drift = policy_drift_summarize(
                chunk_jsons,
                api_key=API_KEY,
                model=DEMO4_MODEL,
                thinking_model=THINKING_MODEL,
                focus_hint=DRIFT_FOCUS,
                canonical_prompt_text=POLICY_PROMPT,
            )
            drift_out.write_text(_json.dumps(drift, indent=2, ensure_ascii=False), encoding="utf-8")

        drifted = drift.get("subjects") or drift.get("drifted_subjects") or []

        # Concatenate per-subject Mermaid timelines so the .mmd file is a single
        # diagram per audio (legacy schema returned one top-level diagram_timeline).
        mmd_blocks = []
        for s in drifted:
            tl = s.get("diagram_timeline")
            if isinstance(tl, str) and tl.strip():
                label = s.get("subject_label") or s.get("subject_id") or "subject"
                mmd_blocks.append(f"%% {label}\n{tl}")
        legacy_tl = drift.get("diagram_timeline")
        if not mmd_blocks and isinstance(legacy_tl, str) and legacy_tl.strip():
            mmd_blocks.append(legacy_tl)
        if mmd_blocks:
            (per_audio_dir / "policy_drift.mmd").write_text(
                "\n\n".join(mmd_blocks), encoding="utf-8"
            )

        meeting_summary = drift.get("meeting_level_summary") or {}
        print(
            f"    drift detector: {len(drifted)} subject(s) tracked across {len(chunk_jsons)} chunks "
            f"→ {drift_out.relative_to(PROCESSED_ROOT)}"
        )
        if meeting_summary.get("headline"):
            print(f"      meeting headline: {meeting_summary['headline'][:140]}")
        for s in drifted[:5]:
            label = s.get("subject_label") or s.get("subject_id") or "?"
            headline = s.get("drift_headline") or s.get("drift_summary") or ""
            stability = (s.get("narrative_stability_assessment") or {}).get("narrative_novelty")
            stability_tag = f" [{stability}]" if stability else ""
            print(f"      · {label}{stability_tag}: {headline[:120]}")
        demo4_report.append(
            {
                "jurisdiction": j.relative_label,
                "audio": str(audio.relative_to(RAW_ROOT)),
                "chunks": len(chunk_jsons),
                "subjects_tracked": len(drifted),
                "subjects_with_drift": sum(1 for s in drifted if s.get("drift_observed")),
                "emergent_value_tensions": meeting_summary.get("emergent_value_tensions") or [],
            }
        )

if demo4_report:
    print(
        f"\nDemo 4 done: {len(demo4_report)} audio file(s) processed. "
        "Gemma's alternating local + global attention kept subject ids consistent "
        "across 15-minute chunks; drift detector reported per-subject drift along the "
        "five narrative_analysis dimensions from policy_analysis_v1.md."
    )
else:
    print("\nDemo 4: no audio files in the inventory. Drop .mp3/.opus/.wav or video (.mp4/.webm/…) under a jurisdiction folder; video is transcoded to Opus before analysis.")


## 9a) Plain transcript *(optional — skip for a 45‑min run)*

Word-for-word transcript per chunk beside Demo 4 JSON. Default language: `en` (`GOVERNANCE_TRANSCRIPTION_LANGUAGES`).


In [ ]:
# Demo 4a — Plain Transcription per audio file.
# Mirrors Demo 4's audio iteration so transcripts land beside the chunk JSONs.
_TRANSCRIPTION_LANGS_RAW = os.environ.get("GOVERNANCE_TRANSCRIPTION_LANGUAGES", "en")
TRANSCRIPTION_LANGS = [
    lang.strip()
    for lang in _TRANSCRIPTION_LANGS_RAW.split(",")
    if lang.strip()
]
# Validate up-front so a typo in the env var fails fast, not after N audio calls.
_SUPPORTED_LANG_TAGS = {tag.lower() for tag in TRANSCRIPTION_SUPPORTED_LANGUAGES.values()}
_invalid_langs = [
    lang for lang in TRANSCRIPTION_LANGS if lang.lower() not in _SUPPORTED_LANG_TAGS
]
if _invalid_langs:
    raise RuntimeError(
        f"GOVERNANCE_TRANSCRIPTION_LANGUAGES contains unsupported tag(s) {_invalid_langs}. "
        f"Supported: {sorted(_SUPPORTED_LANG_TAGS)} (currently {list(TRANSCRIPTION_SUPPORTED_LANGUAGES)})."
    )

_TRANSCRIPTION_MAX_CHUNKS = int(
    os.environ.get("GOVERNANCE_TRANSCRIPTION_MAX_CHUNKS", str(MAX_AUDIO_CHUNKS))
)

demo4a_report = []
_demo4a_force = force_reprocess_outputs()
if not TRANSCRIPTION_LANGS:
    print("Demo 4a: GOVERNANCE_TRANSCRIPTION_LANGUAGES is empty — skipping.")
else:
    print(f"Demo 4a: transcribing in {TRANSCRIPTION_LANGS} (cap {_TRANSCRIPTION_MAX_CHUNKS} chunks/file)")
    for inv in INVENTORIES:
        j = inv.jurisdiction
        audios = inv.audio[:MAX_AUDIO_PER_JUR]
        if not audios:
            continue
        print(f"\n— {j.relative_label}: {len(audios)} audio file(s)")
        for audio in audios:
            print(f"  • {audio.name}")
            per_audio_dir = mirror_output_path(
                input_path=audio, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT, suffix="",
            )
            per_audio_dir.mkdir(parents=True, exist_ok=True)

            rel = audio.resolve().relative_to(RAW_ROOT.resolve())
            scratch = SCRATCH_AUDIO_ROOT / rel.with_suffix("")
            scratch.mkdir(parents=True, exist_ok=True)

            # Reuse Demo 4 chunks if they're already on disk; otherwise chunk now.
            existing = sorted(scratch.glob("chunk_*.mp3"))
            if existing:
                chunks = existing
                print(f"    reusing {len(chunks)} existing chunk(s) from {scratch.name}/")
            else:
                try:
                    chunks = chunk_audio_ffmpeg(audio, out_dir=scratch, chunk_minutes=15, fmt="mp3")
                except Exception as e:
                    print(f"    ! ffmpeg chunking failed: {e}")
                    continue
            chunks = chunks[:_TRANSCRIPTION_MAX_CHUNKS]
            print(f"    {len(chunks)} chunk(s) to transcribe")

            for lang in TRANSCRIPTION_LANGS:
                out_path = per_audio_dir / f"transcript.{lang}.txt"
                if not _demo4a_force and text_output_complete(out_path):
                    print(f"    [{lang}] reuse existing → {out_path.relative_to(PROCESSED_ROOT)}")
                    demo4a_report.append(
                        {
                            "jurisdiction": j.relative_label,
                            "audio": str(audio.relative_to(RAW_ROOT)),
                            "language": lang,
                            "chunks": len(chunks),
                            "chars": len(out_path.read_text(encoding="utf-8")),
                            "errors": 0,
                            "reused": True,
                        }
                    )
                    continue
                pieces = []
                errors = 0
                for idx, chunk_path in enumerate(chunks):
                    try:
                        text = transcribe_audio_with_gemma(
                            api_key=API_KEY,
                            model=GENAI_MODEL,
                            audio_path=chunk_path,
                            language=lang,
                            mime_type=mime_for(chunk_path),
                        )
                    except Exception as e:
                        errors += 1
                        print(f"    ! chunk {idx} [{lang}] failed: {e}")
                        continue
                    if text:
                        pieces.append(text)

                if not pieces:
                    print(f"    [{lang}] no text produced — skipping write")
                    continue

                # Join chunk transcripts with single spaces. The model is instructed
                # not to emit newlines; we still collapse defensively in the helper.
                transcript = " ".join(pieces)
                out_path = per_audio_dir / f"transcript.{lang}.txt"
                out_path.write_text(transcript, encoding="utf-8")
                print(
                    f"    [{lang}] → {out_path.relative_to(PROCESSED_ROOT)} "
                    f"({len(transcript):,} chars from {len(pieces)}/{len(chunks)} chunks"
                    + (f", {errors} error(s)" if errors else "")
                    + ")"
                )
                demo4a_report.append(
                    {
                        "jurisdiction": j.relative_label,
                        "audio": str(audio.relative_to(RAW_ROOT)),
                        "language": lang,
                        "chunks_transcribed": len(pieces),
                        "chunks_attempted": len(chunks),
                        "errors": errors,
                        "chars": len(transcript),
                    }
                )

if demo4a_report:
    print(
        f"\nDemo 4a done: {len(demo4a_report)} transcript file(s) written. "
        "Plain text is now beside each audio's chunk JSONs for citation / human review."
    )
else:
    print("\nDemo 4a: no transcripts written (no audio in inventory or all chunks failed).")

## 10) Contact photos *(optional)*

Classifies `_contact_images/` as person vs. logo/building; writes JSON beside each image.


In [ ]:
# Demo 5 — Contact image enrichment. For each image found under a jurisdiction,
# decide whether it is a person; return perceived demographics or a subject tag.
import json as _json

DEMO5_SYSTEM = (
    "You are a careful image triage assistant for a local-government open-data "
    "pipeline. You analyze public contact photos of elected officials and "
    "department heads scraped from public government websites. You always return "
    "strict JSON only. You never claim certainty about demographic attributes — "
    "every attribute is explicitly perceived from the image and you return null "
    "when the image is too low-resolution, partially occluded, ambiguous, or when "
    "you would be guessing rather than observing."
)
DEMO5_USER = (
    "Look at the attached image and decide whether it depicts a single identifiable "
    "person. Return JSON with this shape:\n\n"
    "{\n"
    "  \"is_person\": bool,\n"
    "  \"confidence\": float between 0.0 and 1.0,\n"
    "  \"perceived\": {\n"
    "    \"age_range\": \"one of: child, teen, 20-29, 30-39, 40-49, 50-59, 60-69, 70-plus, or null\",\n"
    "    \"race\": \"perceived racial category or null\",\n"
    "    \"gender\": \"perceived gender presentation or null\",\n"
    "    \"ethnicity\": \"perceived ethnic background or null\",\n"
    "    \"demeanor\": \"one of: formal, smiling, neutral, stern, candid, or null\"\n"
    "  } or null when is_person is false,\n"
    "  \"subject_tag\": \"short descriptor when is_person is false (e.g. 'city logo', "
    "'aerial photo of courthouse', 'organizational chart'); otherwise null\",\n"
    "  \"notes\": \"short caveat — e.g. 'low resolution', 'side profile', "
    "'multiple people visible', 'official portrait crop'\"\n"
    "}\n\n"
    "Rules:\n"
    "- These are PUBLIC officials in a PUBLIC-records transparency context.\n"
    "- Every 'perceived.*' field is a best visual estimate, not a factual claim.\n"
    "- Return null for any 'perceived' subfield where you would be guessing.\n"
    "- If multiple people appear, set is_person=false and use subject_tag='group photo' (and notes).\n"
    "- Return only the JSON object, no prose or markdown."
)


def _mime_for_image(p):
    ext = p.suffix.lower()
    if ext in (".jpg", ".jpeg"): return "image/jpeg"
    if ext == ".png": return "image/png"
    if ext == ".webp": return "image/webp"
    if ext == ".gif": return "image/gif"
    if ext == ".bmp": return "image/bmp"
    return "application/octet-stream"


MAX_IMAGES_PER_JUR = int(os.environ.get("GOVERNANCE_DEMO_MAX_IMAGES_PER_JUR", "12"))

demo5_report = []
if MAX_IMAGES_PER_JUR <= 0:
    print("\nDemo 5 skipped (Fast scope — pick Standard or Full in §2 for contact images).")
for inv in ([] if MAX_IMAGES_PER_JUR <= 0 else INVENTORIES):
    j = inv.jurisdiction
    if not inv.images:
        continue
    images = inv.images[:MAX_IMAGES_PER_JUR]
    print(f"\n— {j.relative_label}: {len(images)} image(s) (cap={MAX_IMAGES_PER_JUR})")
    for img in images:
        try:
            result = call_google_genai_multimodal(
                api_key=API_KEY,
                model=GENAI_MODEL,
                system_instruction=DEMO5_SYSTEM,
                user_text=DEMO5_USER,
                media=[(img, _mime_for_image(img))],
                temperature=0.0,
                max_output_tokens=1024,
                media_resolution=TOKEN_BUDGET_HIGH,
            )
        except Exception as e:
            print(f"  ! {img.name}: Gemma call failed — {e}")
            continue

        try:
            payload = _json.loads((result.text or "").strip().lstrip("`"))
        except Exception:
            payload = {"_parse_error": True, "_raw": (result.text or "")[:2000]}

        out_path = mirror_output_path(
            input_path=img, raw_root=RAW_ROOT, processed_root=GEMMA_JSON_ROOT,
            suffix=".image_triage.json",
        )
        out_path.write_text(
            _json.dumps(
                {
                    "image": str(img.relative_to(RAW_ROOT)),
                    "jurisdiction_id": j.jurisdiction_id,
                    "model": GENAI_MODEL,
                    "result": payload,
                },
                indent=2, ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        is_person = bool(payload.get("is_person")) if isinstance(payload, dict) else False
        if is_person:
            perceived = payload.get("perceived") or {}
            details = ", ".join(
                f"{k}={v}" for k, v in perceived.items() if v not in (None, "")
            ) or "(no observable attributes)"
            print(f"  · {img.name}: PERSON — {details}")
        else:
            tag = payload.get("subject_tag") if isinstance(payload, dict) else None
            print(f"  · {img.name}: non-person → {tag or '(no tag)'}")
        demo5_report.append(
            {
                "jurisdiction": j.relative_label,
                "jurisdiction_id": j.jurisdiction_id,
                "image": str(img.relative_to(RAW_ROOT)),
                "is_person": is_person,
                "output": str(out_path.relative_to(PROCESSED_ROOT)),
            }
        )

if demo5_report:
    persons = sum(1 for r in demo5_report if r["is_person"])
    print(
        f"\nDemo 5 done: {len(demo5_report)} images classified across "
        f"{len({r['jurisdiction'] for r in demo5_report})} jurisdictions. "
        f"{persons} persons / {len(demo5_report) - persons} non-person subjects."
    )
else:
    print("\nDemo 5: no images in the inventory (check _contact_images/ folders).")

## 11) Merge reference JSON *(optional)*

Joins Gemma outputs with meeting-data / contacts reference files by `jurisdiction_id`.


In [ ]:
# Walk every *.json under GEMMA_JSON_ROOT and attach the matching meeting-data
# and contacts rows by jurisdiction_id, derived from the mirrored output path.
import json as _json


def _jurisdiction_id_from_output(path):
    """Mirror layout is <STATE>/<scope>/<jur_slug>/...  →  jurisdiction_<state>_<scope>_<tail>."""
    try:
        rel = path.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        return None
    parts = rel.parts
    if len(parts) < 3:
        return None
    state, scope, slug = parts[0].lower(), parts[1].lower(), parts[2].lower()
    scope_prefix = f"{scope}_"
    tail = slug[len(scope_prefix):] if slug.startswith(scope_prefix) else slug
    return f"jurisdiction_{state}_{scope}_{tail}"


meeting_data = load_meeting_data_lookup(PIPE.meeting_data_by_jurisdiction_id)
contacts = load_contacts_lookup(PIPE.contacts_by_jurisdiction_id)

if not meeting_data and not contacts:
    print(
        f"No reference lookups found under {PIPE.meeting_data_by_jurisdiction_id} "
        f"or {PIPE.contacts_by_jurisdiction_id} — skip."
    )
else:
    print(
        f"Loaded {len(meeting_data)} meeting_data and {len(contacts)} contacts "
        "rows keyed by jurisdiction_id."
    )
    merged_meeting = 0
    merged_contacts = 0
    missing_both = 0
    skipped = 0
    for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
        if jp.name.startswith("_") or jp.name.startswith("policy_drift"):
            continue
        try:
            data = _json.loads(jp.read_text(encoding="utf-8"))
        except _json.JSONDecodeError:
            skipped += 1
            continue
        jid = _jurisdiction_id_from_output(jp)
        if not jid:
            continue
        analysis = data
        if isinstance(data, dict) and "json_analysis" in data and isinstance(data["json_analysis"], dict):
            analysis = data["json_analysis"]
        if not isinstance(analysis, dict):
            continue
        had_meeting = jid in meeting_data
        had_contacts = jid in contacts
        if not (had_meeting or had_contacts):
            missing_both += 1
            continue
        if had_meeting:
            merge_meeting_data_by_jurisdiction(analysis, meeting_data, jid)
            merged_meeting += 1
        if had_contacts:
            merge_contacts_by_jurisdiction(analysis, contacts, jid)
            merged_contacts += 1
        jp.write_text(_json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
        tags = []
        if had_meeting:
            tags.append("meeting_data")
        if had_contacts:
            tags.append("contacts")
        print(f"  enriched: {jp.relative_to(PROCESSED_ROOT)}  jurisdiction_id={jid}  +{','.join(tags)}")
    print(
        f"\nMerge done: meeting_data={merged_meeting}, contacts={merged_contacts}, "
        f"no-match jurisdictions={missing_both}, unparseable files={skipped}."
    )

## 12) Cross-jurisdiction clustering *(optional)*

EmbeddingGemma finds similar policy language across Tuscaloosa and Big Timber. Needs `GEMINI_API_KEY` and prior demo JSON. Output: `04_embeddings/cross_jurisdiction_pairs.json`.


In [ ]:
# Demo 6 — EmbeddingGemma: cluster similar policy items across jurisdictions.
import json as _json

EMBED_MAX_ITEMS = int(os.environ.get("GOVERNANCE_EMBED_MAX_ITEMS", "200"))
EMBED_SIM_THRESHOLD = float(os.environ.get("GOVERNANCE_EMBED_SIM_THRESHOLD", "0.7"))
EMBED_REPORT_TOP_N = int(os.environ.get("GOVERNANCE_EMBED_REPORT_TOP_N", "30"))


def _harvest_snippets(json_path):
    """Pull high-signal text snippets from one Gemma JSON output."""
    try:
        data = _json.loads(json_path.read_text(encoding="utf-8"))
    except Exception:
        return []
    snippets = []
    payload = data
    if isinstance(data, dict) and isinstance(data.get("json_analysis"), dict):
        payload = data["json_analysis"]
    if not isinstance(payload, dict):
        return []
    # Demo 1 + 2 outputs carry raw_text / headlines.
    if isinstance(payload.get("raw_text"), str) and len(payload["raw_text"]) > 40:
        snippets.append(payload["raw_text"][:600])
    if isinstance(payload.get("headline"), str):
        snippets.append(payload["headline"])
    # Demo 3 outputs: subjects[], decisions[].
    for sub in payload.get("subjects", []) or []:
        name = sub.get("name") or sub.get("descriptive_name") or sub.get("subject_id")
        if name:
            snippets.append(str(name))
    for dec in payload.get("decisions", []) or []:
        title = dec.get("decision_title") or dec.get("title")
        if title:
            snippets.append(str(title))
    return [s.strip() for s in snippets if s and s.strip()]


items = []
for jp in sorted(GEMMA_JSON_ROOT.rglob("*.json")):
    if jp.name.startswith("_"):
        continue
    try:
        rel = jp.resolve().relative_to(GEMMA_JSON_ROOT.resolve())
    except ValueError:
        continue
    parts = rel.parts
    if len(parts) < 3:
        continue
    state, scope, slug = parts[0], parts[1], parts[2]
    jurisdiction = f"{state}/{scope}/{slug}"
    for text in _harvest_snippets(jp):
        items.append({"jurisdiction": jurisdiction, "source": str(rel), "text": text})
    if len(items) >= EMBED_MAX_ITEMS:
        break

items = items[:EMBED_MAX_ITEMS]
print(f"Embedding {len(items)} snippet(s) from {GEMMA_JSON_ROOT}")

if not items:
    print("Demo 6: no JSON snippets to embed yet — run Demos 1–3 first.")
else:
    if not GEMINI_API_KEY:
        raise RuntimeError("Demo 6 needs GEMINI_API_KEY (AI Studio) for EmbeddingGemma, or skip this cell.")
    vectors = embed_text_with_gemma(
        api_key=GEMINI_API_KEY,
        model=EMBEDDING_MODEL,
        texts=[it["text"] for it in items],
    )
    for it, v in zip(items, vectors):
        it["embedding_dims"] = len(v)

    embeddings_root = PIPE.root / "03_processed_outputs" / "04_embeddings"
    embeddings_root.mkdir(parents=True, exist_ok=True)
    items_path = embeddings_root / "items.jsonl"
    items_path.write_text(
        "\n".join(_json.dumps(it, ensure_ascii=False) for it in items) + "\n",
        encoding="utf-8",
    )

    sim = cosine_similarity_matrix(vectors)
    pairs = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i]["jurisdiction"] == items[j]["jurisdiction"]:
                continue  # we want CROSS-jurisdiction matches
            if sim[i][j] < EMBED_SIM_THRESHOLD:
                continue
            pairs.append({
                "similarity": round(float(sim[i][j]), 4),
                "left": {"jurisdiction": items[i]["jurisdiction"], "text": items[i]["text"][:200]},
                "right": {"jurisdiction": items[j]["jurisdiction"], "text": items[j]["text"][:200]},
            })
    pairs.sort(key=lambda p: -p["similarity"])
    pairs = pairs[:EMBED_REPORT_TOP_N]

    pairs_path = embeddings_root / "cross_jurisdiction_pairs.json"
    pairs_path.write_text(
        _json.dumps({"threshold": EMBED_SIM_THRESHOLD, "pairs": pairs}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Demo 6 done — {len(pairs)} cross-jurisdiction pair(s) ≥ {EMBED_SIM_THRESHOLD}.")
    for p in pairs[:5]:
        print(f"  {p['similarity']:.3f}  {p['left']['jurisdiction']}  ↔  {p['right']['jurisdiction']}")
        print(f"            L: {p['left']['text'][:90]}")
        print(f"            R: {p['right']['text'][:90]}")
    print(f"  items   → {items_path.relative_to(PIPE.root)}")
    print(f"  pairs   → {pairs_path.relative_to(PIPE.root)}")


## 13) Safety review *(rerun only — also runs at end of §6)*

ShieldGemma reviews Demo 1–5 LLM outputs → `05_safety_review/`. **On by default** (`GOVERNANCE_SAFETY_REVIEW=1`). Set `0` to skip.


In [ ]:
# Rerun safety review without re-running §6 (same logic as end of pipeline).
from colab_safety_review import run_safety_review, safety_review_enabled

if safety_review_enabled():
    run_safety_review(
        api_key=API_KEY,
        shield_model=SHIELD_MODEL,
        gemma_json_root=GEMMA_JSON_ROOT,
        safety_root=PIPE.root / "03_processed_outputs" / "05_safety_review",
        summaries_root=SUMMARIES_ROOT,
    )
else:
    print("Safety review disabled (GOVERNANCE_SAFETY_REVIEW=0).")


## Troubleshooting (short)

| Issue | Fix |
|-------|-----|
| “Connected to GPU but not utilizing it” | Expected on default **Google AI** path — use **Runtime → CPU**. GPU is only for local HF (`GOVERNANCE_GATEKEEPER_FORCE_HF=1`). |
| Logs on Colab but not on PC Drive | Normal sync lag. Tail **`/content/governance_pipeline_local/00_logs/gatekeeper_*.log`** (mirror, instant) or refresh Drive. |
| What is running now? | Set `GOVERNANCE_PIPELINE_VERBOSE=1` (default). §2 `MEDIA_SCOPE`: `all` \| `pdf` \| `audio` \| `video` (default **video** for MP4 testing). After Gatekeeper: **plan** lists scoped files. Demos 1–3 = PDF; Demo 4 = recordings. |
| Silent 5–10 min / stuck on Demo 3 after one PDF line | Not hung — `gemma-4-31b-it` full-PDF policy analysis is often **3–8 min** per file. Look for `HH:MM:SS ⏳ Calling …` then `✓ Gemma returned`. Per-demo `▶ Demo 3 policy PDFs` completes before Demo 4. `GOVERNANCE_STEP_TIMESTAMPS=1` (default). |
| Looks stuck on Gatekeeper API | AI Studio (1–3 min/file). Log shows `gemma START`. Local HF E2B uses Colab GPU. |
| Full model list spam | Keep `GOVERNANCE_LOG_LLM_CATALOG=0` |
| `count_triageable_files` missing | Runtime restart → re-run sections 1-3 |
| Gatekeeper OOM / CUDA | Only with **local HF** — use a Colab **GPU** runtime and `GOVERNANCE_GATEKEEPER_FORCE_HF=1` |
| Demo 3 empty `.thoughts.md` | Confirm `THINKING_MODEL` resolved to `gemma-4-31b-it`; set `GOVERNANCE_FORCE_THINKING=1` if needed |
| `429 RESOURCE_EXHAUSTED` / page 1 Gemma failed | Auto-retry is in `genai_quota_retry.py` (reload §3). Pace HIGH pages: `GOVERNANCE_GENAI_INTER_CALL_DELAY_HIGH_SECONDS=8`. More retries: `GOVERNANCE_GENAI_QUOTA_RETRIES=8`. Optional: `GOVERNANCE_GENAI_MODEL=gemma-4-31b-it` if listed on your key. |
| COFOG-01 on parks / unclear theme | Open `02_gemma_json/.../*.thinking.theme_audit.json` and `03_human_summaries/.../_meeting_summary.md` (theme table + flags). COFOG follows `primary_theme`; parks should be **Parks and Recreation → COFOG-08**. Re-run Demo 3 with `GOVERNANCE_FORCE_REPROCESS=1`. |
| No video summary / mermaid / one PDF only | Consolidated summary: `03_human_summaries/.../meetings/.../_meeting_summary.md` + `policy_drift.mmd`. Demo 4 + drift need audio in inventory (`audio≥1` in §5). Flat `*.mp4` at jurisdiction root are walked automatically; `_video_assets/*.opus` need the bytes on disk (`GOVERNANCE_INVENTORY_VIDEO_ASSETS=1`, default). Demo 4a for transcripts. |
| `2026_05_06/2026-05-06/` double date | Auto-repair hoists to `meetings/2026_05_06/session/` on organize + consolidated summary. Re-run §6; or set `GOVERNANCE_ORGANIZE_MEETINGS=1` after reload §3. |
| No `_meeting_summary.md` in meeting folder | Default: written beside agenda/minutes **and** under `03_human_summaries/…`. Look for `meetings/…/session/_meeting_summary.md` after repair. Set `GOVERNANCE_CONSOLIDATED_SUMMARY_IN_RAW=0` to only use processed tree. |
| Minutes / audio missing from summary | Demo 3 now runs **per meeting** (agenda + minutes). Demo 4 chunks + drift merged into summary when model accepts audio. Re-run with `GOVERNANCE_FORCE_REPROCESS=1`. |
| `.mp4` not in Demo 4 | Inventory lists video under `audio`; set `GOVERNANCE_DEMO4_VIDEO_CHUNKS=1` (default) for `video/mp4` segments |
| `reuse existing page JSON` / stale outputs | Set `GOVERNANCE_FORCE_REPROCESS=1` and re-run §2→§6, or delete the matching `02_gemma_json/.../page_*.json` / `.thinking.json` |
| Demo 3 `Thinking budget is not supported` | Fixed in code: Gemma gets `include_thoughts` without `thinking_budget`. Re-run §3+§6 with `GOVERNANCE_FORCE_REPROCESS=1` on failed PDFs. |
| Colab looks stuck (no new lines) | Normal on first HF load or each 15‑min chunk (3–15 min). You should see `… still HF weights` / `… still HF generate` every ~45s. Set `GOVERNANCE_HEARTBEAT_SECONDS=30` for faster pings; `0` to disable. |
| Demo 4 fails / no chunks | Need **GPU** + **HF_TOKEN** + `GOVERNANCE_DEMO4_USE_HF=1`. Logs: `ingest opus_15min` then `Hugging Face google/gemma-4-E2B-it`. Opus/ffmpeg reuse `_scratch_audio_chunks` on Drive. Set `GOVERNANCE_DEMO4_USE_HF=0` only to try Google API (Edge models often missing). |
| `no such file` … `2026-02-18/agenda/` | Stale paths after `session/` repair. Re-run **§6** (reload + remap). Files live under `meetings/2026_02_18/session/agenda/`. |
| `meeting brief LLM failed: NoneType ... models` | Reload §3 after pull; brief now builds its own GenAI client. |
| No data root | Sync pilots with `01_copy_scraped_meetings_cache_to_gdrive.py` |


## Where to look on Drive (after Run all)

```
03_processed_outputs/
  00_logs/triage_report_*.json         ← what was kept vs excluded
  02_gemma_json/                       ← Demo 1–4 JSON + sidecars
  03_human_summaries/                  ← Smart Brevity (if generated)
```

**Live demo tip:** open `00_logs/gatekeeper_*.log` on Drive during §6. Before API calls you should see walk/classify lines; during triage, `gemma START` / `gemma DONE`.
